# Module 06 — Differential Expression Analysis

This notebook visualizes pre-computed pseudobulk differential expression results
from `scripts/06_differential.py`. Pseudobulk aggregation (summing counts per
donor per cell type) avoids treating cells as independent observations, and
DESeq2/pyDESeq2 is used for statistical testing.

**Key findings:**
- **21 powered comparisons** (cell type x condition pairs with >= 3 donors per group)
- **949 unique significant genes** (|log2FC| > 0.5, padj < 0.05)
- Herniated samples excluded from primary analysis (confounded with acute trauma)
- NP_fibrocartilaginous and EP_hyaline are newly annotated cell types with DE results

**Data sources:**
- `results/differential/de_summary_table.tsv` — Summary of all comparisons
- `results/differential/skipped_comparisons.tsv` — Underpowered comparisons
- `results/differential/de_results_combined.tsv` — Full DE results
- `results/differential/volcano_plots/` — Volcano plots per comparison
- `results/differential/heatmaps/` — Heatmaps of top DE genes

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 200, 'savefig.bbox': 'tight'})

# ── Paths ──────────────────────────────────────────────────────────────────
BASE = Path('..').resolve()
RESULTS = BASE / 'results' / 'differential'
VOLCANO_DIR = RESULTS / 'volcano_plots'
HEATMAP_DIR = RESULTS / 'heatmaps'

print(f'Results directory: {RESULTS}')
print(f'Volcano plots:    {len(list(VOLCANO_DIR.glob("*.png")))} files')
print(f'Heatmaps:         {len(list(HEATMAP_DIR.glob("*.png")))} files')

## DE Summary Table

Each row represents one powered comparison (cell type x condition pair). The
table shows the number of significantly up- and down-regulated genes per
comparison.

In [ ]:
summary = pd.read_csv(RESULTS / 'de_summary_table.tsv', sep='\t')
print(f'Total powered comparisons: {len(summary)}')
print(f'Total significant genes (sum across comparisons): {summary["n_total"].sum()}')

display(summary.style.set_caption('Pseudobulk DE Summary — All Powered Comparisons'))

In [ ]:
# Unique significant genes across all comparisons
de_combined = pd.read_csv(RESULTS / 'de_results_combined.tsv', sep='\t')
sig = de_combined[(de_combined['padj'] < 0.05) & (de_combined['log2FC'].abs() > 0.5)]
print(f'Unique significant genes: {sig["gene"].nunique()}')
print(f'Total significant hits:   {len(sig)}')
print(f'\nCell types with DE results: {sig["cell_type"].nunique()}')
for ct in sorted(sig['cell_type'].unique()):
    n = sig[sig['cell_type'] == ct]['gene'].nunique()
    print(f'  {ct}: {n} genes')

## Skipped (Underpowered) Comparisons

Comparisons that did not meet the minimum donor threshold (>= 3 donors per
group) are listed here with the reason for exclusion.

In [ ]:
skipped = pd.read_csv(RESULTS / 'skipped_comparisons.tsv', sep='\t')
print(f'Skipped comparisons: {len(skipped)}')
display(skipped)

## Volcano Plots

Volcano plots show the relationship between fold change (x-axis) and
statistical significance (y-axis) for each comparison. Points colored by
significance threshold (|log2FC| > 0.5, padj < 0.05).

### Key resident cell type comparisons

In [ ]:
# Display key volcano plots for resident cell types
key_volcanos = [
    'volcano_NP_mature_chondrocyte_healthy_vs_degenerated_all.png',
    'volcano_NP_fibrocartilaginous_healthy_vs_degenerated_all.png',
    'volcano_AF_outer_healthy_vs_degenerated_all.png',
    'volcano_AF_inner_healthy_vs_degenerated_all.png',
    'volcano_EP_hyaline_healthy_vs_degenerated_all.png',
    'volcano_Endothelial_cells_healthy_vs_degenerated_all.png',
]

for fname in key_volcanos:
    fpath = VOLCANO_DIR / fname
    if fpath.exists():
        label = fname.replace('volcano_', '').replace('.png', '').replace('_', ' ')
        display(Markdown(f'**{label}**'))
        display(Image(filename=str(fpath), width=700))
    else:
        print(f'Not found: {fname}')

### Severity-stratified comparisons (mild vs severe)

In [ ]:
# Display mild vs severe comparisons
severity_volcanos = sorted(VOLCANO_DIR.glob('*mild_vs_severe*.png'))

for fpath in severity_volcanos:
    label = fpath.stem.replace('volcano_', '').replace('_', ' ')
    display(Markdown(f'**{label}**'))
    display(Image(filename=str(fpath), width=700))

### All remaining volcano plots

In [ ]:
# Show any remaining volcano plots not already displayed
already_shown = set(key_volcanos + [f.name for f in severity_volcanos])
remaining = sorted(f for f in VOLCANO_DIR.glob('*.png') if f.name not in already_shown)

print(f'Remaining volcano plots: {len(remaining)}')
for fpath in remaining:
    label = fpath.stem.replace('volcano_', '').replace('_', ' ')
    display(Markdown(f'**{label}**'))
    display(Image(filename=str(fpath), width=700))

## Heatmaps — Top DE Genes

Heatmaps show expression of the top differentially expressed genes across
pseudobulk samples (donors), organized by condition. These complement the
volcano plots by showing the consistency of expression changes across donors.

In [ ]:
heatmap_files = sorted(HEATMAP_DIR.glob('*.png'))
print(f'Heatmap files: {len(heatmap_files)}')

for fpath in heatmap_files:
    label = fpath.stem.replace('heatmap_', '').replace('_', ' ')
    display(Markdown(f'**{label}**'))
    display(Image(filename=str(fpath), width=800))

## Composition Analysis

Cell type proportion changes between conditions, tested with permutation-based
methods.

In [ ]:
comp_path = RESULTS / 'composition_analysis.tsv'
if comp_path.exists():
    composition = pd.read_csv(comp_path, sep='\t')
    display(composition.style.set_caption('Cell Type Composition Changes'))
else:
    print('Composition analysis file not found')

## Status — Module 06 Complete

### Summary
- **21 powered comparisons** across multiple cell types and conditions
- **949 unique significant genes** identified
- Pseudobulk approach correctly treats donors (not cells) as biological replicates
- Herniated condition excluded from primary healthy-vs-degenerated contrasts
- NP_fibrocartilaginous and EP_hyaline represent newly annotated cell types
  with substantial DE signal

### Key observations
- NP_mature_chondrocyte and NP_fibrocartilaginous show the most DE genes
- AF_outer has more DE genes than AF_inner (larger sample size)
- Severity-stratified analyses (mild vs severe) reveal dose-dependent effects
- Non-resident cells (T cells, macrophages) show immune activation signatures